# Teste métricas de clusters

In [20]:
from sqlalchemy import Engine, MetaData, Table, cast, func, update
from sqlalchemy.sql.sqltypes import BigInteger, DOUBLE_PRECISION, Integer, SmallInteger, Numeric, Double

import os
from pathlib import Path
import pandas as pd


from sql_utils import drop_table, build_postgres_engine


project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )
    
chunck_size = 100_000 # Se estiver estourando mt ram, abaixe esse numero
max_clusters = 30

schema = "norm_por_1k"

In [2]:
import pandas as pd
import numpy as np


def filter_rows_by_coordinate_sum(
    df: pd.DataFrame,
    reference_column: str,
    coordinate_columns: list[str],
    expected_sum: float = 1000,
    sum_tolerance: float = 1e-6,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Separa o DataFrame em:
    - valid_df: linhas em que as colunas de coordenadas somam expected_sum;
    - excluded_df: linhas excluídas, com a soma calculada.

    Use isso antes de passar o DataFrame para o test_gmm.
    """

    required_columns = [reference_column, *coordinate_columns]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"As seguintes colunas não existem no DataFrame: {missing_columns}"
        )

    coordinate_sum = df[coordinate_columns].sum(axis=1)

    valid_mask = np.isclose(
        coordinate_sum,
        expected_sum,
        atol=sum_tolerance,
        rtol=0,
    )

    valid_df = df.loc[valid_mask].copy()

    excluded_df = df.loc[~valid_mask, [reference_column, *coordinate_columns]].copy()
    excluded_df["coordinate_sum"] = coordinate_sum.loc[~valid_mask]

    return valid_df, excluded_df

In [3]:
# import pandas as pd
# from sklearn.mixture import GaussianMixture

# # Useri como base, gerado pelo chat gpt

# def test_gmm(X: pd.DataFrame, k_min=2, k_max=12) -> pd.DataFrame:
#     results = []

#     for k in range(k_min, k_max + 1):
#         model = GaussianMixture(
#             n_components=k,
#             covariance_type="full",
#             random_state=42,
#             n_init=5,
#         )

#         model.fit(X)
#         labels = model.predict(X)

#         results.append({
#             "k": k,
#             "bic": model.bic(X),
#             "aic": model.aic(X),
#             "min_cluster_size": pd.Series(labels).value_counts().min(),
#             "max_cluster_size": pd.Series(labels).value_counts().max(),
#         })

#     return pd.DataFrame(results)



In [4]:
import pandas as pd
from sklearn.mixture import GaussianMixture


def test_gmm(
    df: pd.DataFrame,
    reference_column: str,
    coordinate_columns: list[str],
    k_min: int = 2,
    k_max: int = 12,
    selected_k: int | None = None,
    random_state: int = 42,
    n_init: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Testa modelos Gaussian Mixture Model para diferentes valores de k
    e retorna:
    
    1. Um DataFrame com estatísticas de ajuste para cada k.
    2. Um DataFrame relacionando a coluna de referência ao cluster atribuído.

    Parâmetros
    ----------
    df:
        DataFrame original contendo a coluna de referência e as colunas numéricas.
    
    reference_column:
        Nome da coluna usada como identificador, por exemplo:
        'cod_municipio', 'cod_estado', 'id_tabela'.
    
    coordinate_columns:
        Lista de colunas numéricas usadas para ajustar o GMM.
    
    k_min:
        Número mínimo de clusters testado.
    
    k_max:
        Número máximo de clusters testado.
    
    selected_k:
        Número de clusters usado para gerar o DataFrame final de classificação.
        Se None, usa o k com menor BIC.
    
    random_state:
        Semente de aleatoriedade.
    
    n_init:
        Número de inicializações do GMM.

    Retorna
    -------
    stats_df:
        DataFrame com AIC, BIC e tamanhos dos clusters para cada k.
    
    clusters_df:
        DataFrame com a coluna de referência e o cluster atribuído.
    """

    missing_columns = [
        column
        for column in [reference_column, *coordinate_columns]
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(f"As seguintes colunas não existem no DataFrame: {missing_columns}")

    X = df[coordinate_columns].copy()

    if X.isna().any().any():
        raise ValueError("As colunas de coordenadas possuem valores nulos. Trate os NaNs antes de ajustar o GMM.")

    results = []
    fitted_models = {}

    for k in range(k_min, k_max + 1):

        print(f"testando para {k} clusters")
        model = GaussianMixture(
            n_components=k,
            covariance_type="full",
            random_state=random_state,
            n_init=n_init,
        )

        model.fit(X)
        labels = model.predict(X)

        cluster_counts = pd.Series(labels).value_counts()

        results.append({
            "k": k,
            "bic": model.bic(X),
            "aic": model.aic(X),
            "min_cluster_size": cluster_counts.min(),
            "max_cluster_size": cluster_counts.max(),
            "mean_cluster_size": cluster_counts.mean(),
        })

        fitted_models[k] = model

    stats_df = pd.DataFrame(results)

    if selected_k is None:
        selected_k = int(stats_df.loc[stats_df["bic"].idxmin(), "k"])

    if selected_k not in fitted_models:
        raise ValueError(
            f"selected_k={selected_k} não foi testado. "
            f"Escolha um valor entre {k_min} e {k_max}."
        )

    final_model = fitted_models[selected_k]
    final_labels = final_model.predict(X)

    clusters_df = df[[reference_column]].copy()
    clusters_df["cluster"] = final_labels
    clusters_df["k"] = selected_k

    return stats_df, clusters_df

## Load dist_idade

In [ ]:
dist_idade = pd.read_sql_table("dist_idade", con=engine, schema=schema)
dist_idade = dist_idade.fillna(0)

dist_idade, excluidos = filter_rows_by_coordinate_sum(
    df = dist_idade, 
    reference_column="cod_territorio",
    coordinate_columns=['0_4', '10_14',
       '15_19', '20_24', '25_29', '30_34', '35_39', '40_44', '45_49', '50_54',
       '55_59', '5_9', '60_64', '65_69', '70_74', '75_79', '80_84', '85_89',
       '90_94', '95_99', '100_mais'],
    sum_tolerance= 1e-4
)

dist_idade = dist_idade[dist_idade["cod_territorio"].astype(int) > 100].copy()

In [27]:
dist_idade_total = dist_idade[dist_idade["populacao"] == "total"].copy()
dist_idade_restante = dist_idade[dist_idade["populacao"] != "total"].copy()

In [21]:
import pandas as pd

def pivot_multiplas_colunas_populacao(
    df: pd.DataFrame,
    categoria_col: str,
    populacao_cols: list[str],
    rename_categories: dict | None = None,
    aggfunc: str = "sum",
    fill_value=0,
) -> pd.DataFrame:
    """
    Pivot várias colunas de população ao mesmo tempo.

    Preserva todas as colunas que NÃO sejam:
    - a coluna de categoria usada no pivot;
    - as colunas de população listadas em populacao_cols.

    Parâmetros
    ----------
    df:
        DataFrame original.

    categoria_col:
        Coluna que separa as linhas.
        Exemplo: "sexo", "genero", "tipo_populacao".

    populacao_cols:
        Lista de colunas populacionais que devem ser pivotadas.
        Exemplo: ["pop_0_4", "pop_5_9", "pop_10_14"].

    rename_categories:
        Dicionário opcional para renomear valores da categoria.
        Exemplo:
        {
            "Homens": "homens",
            "Mulheres": "mulheres"
        }

    aggfunc:
        Função de agregação caso exista mais de uma linha para o mesmo grupo.
        Normalmente "sum".

    fill_value:
        Valor usado quando uma combinação não existir.
    """

    if rename_categories is None:
        rename_categories = {}

    # Colunas de identificação que serão preservadas
    id_cols = [
        col for col in df.columns
        if col not in [categoria_col, *populacao_cols]
    ]

    # Faz o pivot com múltiplas colunas de valores
    pivot = df.pivot_table(
        index=id_cols,
        columns=categoria_col,
        values=populacao_cols,
        aggfunc=aggfunc,
        fill_value=fill_value,
    )

    # Achata o MultiIndex das colunas
    # Exemplo: ("pop_0_4", "Homens") -> "pop_0_4_homens"
    pivot.columns = [
        f"{pop_col}_{rename_categories.get(categoria, str(categoria))}"
        for pop_col, categoria in pivot.columns
    ]

    pivot = pivot.reset_index()

    return pivot

In [30]:
populacao_cols = [
    '0_4', '10_14',
    '15_19', '20_24', '25_29', '30_34', '35_39', '40_44', '45_49', '50_54',
    '55_59', '5_9', '60_64', '65_69', '70_74', '75_79', '80_84', '85_89',
    '90_94', '95_99', '100_mais'
]

dist_idade_gen = pivot_multiplas_colunas_populacao(
    df=dist_idade_restante,
    categoria_col='populacao',
    populacao_cols=populacao_cols,
    rename_categories={
        "Homens": "homens",
        "Mulheres": "mulheres",
    },
)

In [7]:
dist_idade.columns

Index(['id', 'cod_territorio', 'territorio', 'populacao', '0_4', '10_14',
       '15_19', '20_24', '25_29', '30_34', '35_39', '40_44', '45_49', '50_54',
       '55_59', '5_9', '60_64', '65_69', '70_74', '75_79', '80_84', '85_89',
       '90_94', '95_99', '100_mais'],
      dtype='str')

testando pelo total

In [8]:
stats, custers = test_gmm(
   df = dist_idade_total, 
   reference_column="cod_territorio",
   coordinate_columns=['0_4', '10_14',
      '15_19', '20_24', '25_29', '30_34', '35_39', '40_44', '45_49', '50_54',
      '55_59', '5_9', '60_64', '65_69', '70_74', '75_79', '80_84', '85_89',
      '90_94', '95_99', '100_mais'],
   k_min=2, k_max=max_clusters
   )

testando para 2 clusters
testando para 3 clusters
testando para 4 clusters
testando para 5 clusters
testando para 6 clusters
testando para 7 clusters
testando para 8 clusters
testando para 9 clusters
testando para 10 clusters
testando para 11 clusters
testando para 12 clusters
testando para 13 clusters
testando para 14 clusters
testando para 15 clusters
testando para 16 clusters
testando para 17 clusters
testando para 18 clusters
testando para 19 clusters
testando para 20 clusters


In [19]:
best_bic_row = stats.loc[stats["bic"].idxmin()]
best_aic_row = stats.loc[stats["aic"].idxmin()]

print(f"menor BIC: {best_bic_row['bic']} | k = {best_bic_row['k']}")
print(f"menor AIC: {best_aic_row['aic']} | k = {best_aic_row['k']}")

stats.sort_values("aic")

menor BIC: 264962.42618618417 | k = 3.0
menor AIC: 254099.9844062482 | k = 20.0


,k,bic,aic,min_cluster_size,max_cluster_size,mean_cluster_size
18,20,285231.236450,254099.984406,14,439,173.800000
12,14,276515.751561,254725.721221,78,525,248.285714
17,19,284339.541640,254765.159881,15,344,182.947368
13,15,278139.812282,254792.911659,43,522,231.733333
15,17,281515.399812,255054.758620,27,589,204.470588
14,16,279967.715726,255063.944818,45,426,217.250000
16,18,283128.483816,255110.972341,50,463,193.111111
11,13,275356.569981,255123.409926,77,552,267.384615
7,9,269433.718063,255428.039144,71,759,386.222222
9,11,272646.258015,255526.838528,79,647,316.000000


In [32]:
dist_idade_gen.columns

Index(['id', 'cod_territorio', 'territorio', '0_4_homens', '0_4_mulheres',
       '100_mais_homens', '100_mais_mulheres', '10_14_homens',
       '10_14_mulheres', '15_19_homens', '15_19_mulheres', '20_24_homens',
       '20_24_mulheres', '25_29_homens', '25_29_mulheres', '30_34_homens',
       '30_34_mulheres', '35_39_homens', '35_39_mulheres', '40_44_homens',
       '40_44_mulheres', '45_49_homens', '45_49_mulheres', '50_54_homens',
       '50_54_mulheres', '55_59_homens', '55_59_mulheres', '5_9_homens',
       '5_9_mulheres', '60_64_homens', '60_64_mulheres', '65_69_homens',
       '65_69_mulheres', '70_74_homens', '70_74_mulheres', '75_79_homens',
       '75_79_mulheres', '80_84_homens', '80_84_mulheres', '85_89_homens',
       '85_89_mulheres', '90_94_homens', '90_94_mulheres', '95_99_homens',
       '95_99_mulheres'],
      dtype='str')

In [33]:
stats, custers = test_gmm(
   df = dist_idade_gen, 
   reference_column="cod_territorio",
   coordinate_columns=['0_4_homens', '0_4_mulheres',
       '100_mais_homens', '100_mais_mulheres', '10_14_homens',
       '10_14_mulheres', '15_19_homens', '15_19_mulheres', '20_24_homens',
       '20_24_mulheres', '25_29_homens', '25_29_mulheres', '30_34_homens',
       '30_34_mulheres', '35_39_homens', '35_39_mulheres', '40_44_homens',
       '40_44_mulheres', '45_49_homens', '45_49_mulheres', '50_54_homens',
       '50_54_mulheres', '55_59_homens', '55_59_mulheres', '5_9_homens',
       '5_9_mulheres', '60_64_homens', '60_64_mulheres', '65_69_homens',
       '65_69_mulheres', '70_74_homens', '70_74_mulheres', '75_79_homens',
       '75_79_mulheres', '80_84_homens', '80_84_mulheres', '85_89_homens',
       '85_89_mulheres', '90_94_homens', '90_94_mulheres', '95_99_homens',
       '95_99_mulheres'],
   k_min=2, k_max=max_clusters
   )

testando para 2 clusters
testando para 3 clusters
testando para 4 clusters
testando para 5 clusters
testando para 6 clusters
testando para 7 clusters
testando para 8 clusters
testando para 9 clusters
testando para 10 clusters
testando para 11 clusters
testando para 12 clusters
testando para 13 clusters
testando para 14 clusters
testando para 15 clusters
testando para 16 clusters
testando para 17 clusters
testando para 18 clusters
testando para 19 clusters
testando para 20 clusters
testando para 21 clusters
testando para 22 clusters
testando para 23 clusters
testando para 24 clusters
testando para 25 clusters
testando para 26 clusters
testando para 27 clusters
testando para 28 clusters
testando para 29 clusters
testando para 30 clusters


In [34]:
best_bic_row = stats.loc[stats["bic"].idxmin()]
best_aic_row = stats.loc[stats["aic"].idxmin()]

print(f"menor BIC: {best_bic_row['bic']} | k = {best_bic_row['k']}")
print(f"menor AIC: {best_aic_row['aic']} | k = {best_aic_row['k']}")

stats.sort_values("aic")

menor BIC: -576947.1746652153 | k = 2.0
menor AIC: -593028.3990586596 | k = 6.0


,k,bic,aic,min_cluster_size,max_cluster_size,mean_cluster_size
4,6,-558198.677723,-593028.399059,279,835,570.000000
3,5,-563738.466930,-592762.211810,472,1031,684.000000
2,4,-569220.182093,-592437.950518,683,1027,855.000000
5,7,-551505.490720,-592141.188511,108,743,488.571429
6,8,-544887.922462,-591329.596708,112,736,427.500000
1,3,-573025.254543,-590437.046513,801,1710,1140.000000
7,9,-538134.992021,-590382.642722,125,548,380.000000
8,10,-531447.173019,-589500.800175,109,562,342.000000
0,2,-576947.174665,-588552.990180,1710,1710,1710.000000
9,11,-524154.365959,-588013.969571,77,536,310.909091


## dist_renda_ibge

#

In [ ]:
dist_renda = pd.read_sql_table("dist_renda_T", con=engine, schema=schema)

dist_renda = dist_renda.fillna(0)

dist_renda = dist_renda.drop(columns=["Total"])

In [54]:
dist_renda

,id,Cod,"Brasil, Unidade da Federação e Município",grupo,Até 1/4 de salário mínimo,Mais de 1 a 2 salários mínimos,Mais de 1/2 a 1 salário mínimo,Mais de 1/4 a 1/2 salário mínimo,Mais de 10 a 15 salários mínimos,Mais de 15 a 20 salários mínimos,Mais de 2 a 3 salários mínimos,Mais de 20 salários mínimos,Mais de 3 a 5 salários mínimos,Mais de 5 a 10 salários mínimos,Sem rendimento
0,0,1,Brasil,Homens,33.523582,326.437221,226.947442,53.741396,14.494516,8.317215,157.363922,8.807758,107.311190,56.432394,6.623364
1,1,1,Brasil,Mulheres,45.102610,328.551107,260.715225,75.414123,9.383057,4.647824,122.870931,3.989496,92.402876,43.545570,13.377207
2,2,1,Brasil,Total,38.574070,327.359231,241.676120,63.194508,12.265023,6.716716,142.318922,6.706150,100.808559,50.811476,9.569225
3,3,11,Rondônia,Homens,19.115232,350.969218,235.981291,43.874579,9.095372,5.237937,162.126872,4.757004,114.055963,44.993434,9.795591
4,4,11,Rondônia,Mulheres,41.745710,317.067476,274.329471,79.157425,5.650975,2.363997,122.690725,2.396830,99.685530,33.099609,21.812252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5209,5209,5221858,Valparaíso de Goiás (GO),Mulheres,22.653504,399.883599,250.061558,41.523963,3.603967,0.514852,145.770376,0.447698,101.694536,26.839478,7.028854
5210,5210,5221858,Valparaíso de Goiás (GO),Total,15.620565,403.225479,209.970401,32.680534,5.504197,1.013664,176.620849,1.054211,111.898390,38.205003,4.206706
5211,5211,5300108,Brasília (DF),Homens,12.622670,281.254210,165.021637,28.540788,46.145700,24.643105,145.468557,26.560285,131.126758,132.862055,5.752887
5212,5212,5300108,Brasília (DF),Mulheres,21.599967,276.410924,210.579644,48.058907,32.258552,16.757631,116.653132,14.103570,136.793744,115.647462,11.136467


In [ ]:
dist_renda.columns

Index(['id', 'Cod', 'Brasil, Unidade da Federação e Município', 'grupo',
       'Até 1/4 de salário mínimo', 'Mais de 1 a 2 salários mínimos',
       'Mais de 1/2 a 1 salário mínimo', 'Mais de 1/4 a 1/2 salário mínimo',
       'Mais de 10 a 15 salários mínimos', 'Mais de 15 a 20 salários mínimos',
       'Mais de 2 a 3 salários mínimos', 'Mais de 20 salários mínimos',
       'Mais de 3 a 5 salários mínimos', 'Mais de 5 a 10 salários mínimos',
       'Sem rendimento', 'Total'],
      dtype='str')

In [55]:


dist_renda, excluidos = filter_rows_by_coordinate_sum(
    df = dist_renda, 
    reference_column="Cod",
    coordinate_columns=['Até 1/4 de salário mínimo', 'Mais de 1 a 2 salários mínimos',
       'Mais de 1/2 a 1 salário mínimo', 'Mais de 1/4 a 1/2 salário mínimo',
       'Mais de 10 a 15 salários mínimos', 'Mais de 15 a 20 salários mínimos',
       'Mais de 2 a 3 salários mínimos', 'Mais de 20 salários mínimos',
       'Mais de 3 a 5 salários mínimos', 'Mais de 5 a 10 salários mínimos',
       'Sem rendimento'],
    sum_tolerance= 2
)
excluidos

,Cod,Até 1/4 de salário mínimo,Mais de 1 a 2 salários mínimos,Mais de 1/2 a 1 salário mínimo,Mais de 1/4 a 1/2 salário mínimo,Mais de 10 a 15 salários mínimos,Mais de 15 a 20 salários mínimos,Mais de 2 a 3 salários mínimos,Mais de 20 salários mínimos,Mais de 3 a 5 salários mínimos,Mais de 5 a 10 salários mínimos,Sem rendimento,coordinate_sum


In [62]:
dist_renda_total = dist_renda[dist_renda["grupo"] == "Total"].copy()
dist_renda_restante = dist_renda[dist_renda["grupo"] != "Total"].copy()

In [63]:
dist_renda_gen = pivot_multiplas_colunas_populacao(
    df= dist_renda_restante,
    categoria_col="grupo",
    populacao_cols=['Até 1/4 de salário mínimo', 'Mais de 1 a 2 salários mínimos',
       'Mais de 1/2 a 1 salário mínimo', 'Mais de 1/4 a 1/2 salário mínimo',
       'Mais de 10 a 15 salários mínimos', 'Mais de 15 a 20 salários mínimos',
       'Mais de 2 a 3 salários mínimos', 'Mais de 20 salários mínimos',
       'Mais de 3 a 5 salários mínimos', 'Mais de 5 a 10 salários mínimos',
       'Sem rendimento'],
    rename_categories={
        "Homens": "homens",
        "Mulheres": "mulheres",
    },
)

In [64]:
dist_renda_gen.columns

Index(['id', 'Cod', 'Brasil, Unidade da Federação e Município',
       'Até 1/4 de salário mínimo_homens',
       'Até 1/4 de salário mínimo_mulheres',
       'Mais de 1 a 2 salários mínimos_homens',
       'Mais de 1 a 2 salários mínimos_mulheres',
       'Mais de 1/2 a 1 salário mínimo_homens',
       'Mais de 1/2 a 1 salário mínimo_mulheres',
       'Mais de 1/4 a 1/2 salário mínimo_homens',
       'Mais de 1/4 a 1/2 salário mínimo_mulheres',
       'Mais de 10 a 15 salários mínimos_homens',
       'Mais de 10 a 15 salários mínimos_mulheres',
       'Mais de 15 a 20 salários mínimos_homens',
       'Mais de 15 a 20 salários mínimos_mulheres',
       'Mais de 2 a 3 salários mínimos_homens',
       'Mais de 2 a 3 salários mínimos_mulheres',
       'Mais de 20 salários mínimos_homens',
       'Mais de 20 salários mínimos_mulheres',
       'Mais de 3 a 5 salários mínimos_homens',
       'Mais de 3 a 5 salários mínimos_mulheres',
       'Mais de 5 a 10 salários mínimos_homens',
       'M

In [65]:
stats, custers = test_gmm(
   df = dist_renda_total, 
   reference_column="Cod",
   coordinate_columns=['Até 1/4 de salário mínimo', 'Mais de 1 a 2 salários mínimos',
       'Mais de 1/2 a 1 salário mínimo', 'Mais de 1/4 a 1/2 salário mínimo',
       'Mais de 10 a 15 salários mínimos', 'Mais de 15 a 20 salários mínimos',
       'Mais de 2 a 3 salários mínimos', 'Mais de 20 salários mínimos',
       'Mais de 3 a 5 salários mínimos', 'Mais de 5 a 10 salários mínimos',
       'Sem rendimento'],
   k_min=2, k_max=max_clusters
)

best_bic_row = stats.loc[stats["bic"].idxmin()]
best_aic_row = stats.loc[stats["aic"].idxmin()]

print(f"menor BIC: {best_bic_row['bic']} | k = {best_bic_row['k']}")
print(f"menor AIC: {best_aic_row['aic']} | k = {best_aic_row['k']}")

stats.sort_values("aic")

testando para 2 clusters
testando para 3 clusters
testando para 4 clusters
testando para 5 clusters
testando para 6 clusters
testando para 7 clusters
testando para 8 clusters
testando para 9 clusters
testando para 10 clusters
testando para 11 clusters
testando para 12 clusters
testando para 13 clusters
testando para 14 clusters
testando para 15 clusters
testando para 16 clusters
testando para 17 clusters
testando para 18 clusters
testando para 19 clusters
testando para 20 clusters
testando para 21 clusters
testando para 22 clusters
testando para 23 clusters
testando para 24 clusters


/usr/local/lib/python3.14/site-packages/sklearn/mixture/_base.py:293: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


testando para 25 clusters
testando para 26 clusters
testando para 27 clusters
testando para 28 clusters
testando para 29 clusters
testando para 30 clusters
menor BIC: 117863.41759762166 | k = 11.0
menor AIC: 112709.61356003738 | k = 18.0


,k,bic,aic,min_cluster_size,max_cluster_size,mean_cluster_size
16,18,120370.681459,112709.613560,17,291,96.555556
18,20,121239.776534,112726.872147,18,294,86.900000
26,28,124706.041662,112785.791324,11,288,62.071429
12,14,118758.400143,112801.005220,28,305,124.142857
15,17,120062.777689,112827.628034,16,292,102.235294
20,22,122200.200571,112835.459696,12,300,79.000000
21,23,122634.079440,112843.420321,20,291,75.565217
14,16,119655.210814,112845.979402,17,293,108.625000
11,13,118386.216463,112854.739783,25,300,133.692308
24,26,123931.032774,112862.618924,12,293,66.846154


In [67]:
stats, custers = test_gmm(
   df = dist_renda_gen, 
   reference_column="Cod",
   coordinate_columns=['Até 1/4 de salário mínimo_homens',
       'Até 1/4 de salário mínimo_mulheres',
       'Mais de 1 a 2 salários mínimos_homens',
       'Mais de 1 a 2 salários mínimos_mulheres',
       'Mais de 1/2 a 1 salário mínimo_homens',
       'Mais de 1/2 a 1 salário mínimo_mulheres',
       'Mais de 1/4 a 1/2 salário mínimo_homens',
       'Mais de 1/4 a 1/2 salário mínimo_mulheres',
       'Mais de 10 a 15 salários mínimos_homens',
       'Mais de 10 a 15 salários mínimos_mulheres',
       'Mais de 15 a 20 salários mínimos_homens',
       'Mais de 15 a 20 salários mínimos_mulheres',
       'Mais de 2 a 3 salários mínimos_homens',
       'Mais de 2 a 3 salários mínimos_mulheres',
       'Mais de 20 salários mínimos_homens',
       'Mais de 20 salários mínimos_mulheres',
       'Mais de 3 a 5 salários mínimos_homens',
       'Mais de 3 a 5 salários mínimos_mulheres',
       'Mais de 5 a 10 salários mínimos_homens',
       'Mais de 5 a 10 salários mínimos_mulheres', 'Sem rendimento_homens',
       'Sem rendimento_mulheres'],
   k_min=2, k_max=max_clusters
)

best_bic_row = stats.loc[stats["bic"].idxmin()]
best_aic_row = stats.loc[stats["aic"].idxmin()]

print(f"menor BIC: {best_bic_row['bic']} | k = {best_bic_row['k']}")
print(f"menor AIC: {best_aic_row['aic']} | k = {best_aic_row['k']}")

stats.sort_values("aic")

testando para 2 clusters
testando para 3 clusters
testando para 4 clusters
testando para 5 clusters
testando para 6 clusters
testando para 7 clusters
testando para 8 clusters
testando para 9 clusters
testando para 10 clusters
testando para 11 clusters
testando para 12 clusters
testando para 13 clusters
testando para 14 clusters
testando para 15 clusters
testando para 16 clusters
testando para 17 clusters
testando para 18 clusters
testando para 19 clusters
testando para 20 clusters
testando para 21 clusters
testando para 22 clusters
testando para 23 clusters
testando para 24 clusters
testando para 25 clusters
testando para 26 clusters
testando para 27 clusters
testando para 28 clusters
testando para 29 clusters
testando para 30 clusters
menor BIC: -194810.23327200292 | k = 6.0
menor AIC: -218983.0361354256 | k = 20.0


,k,bic,aic,min_cluster_size,max_cluster_size,mean_cluster_size
18,20,-185021.110848,-218983.036135,22,529,173.800000
23,25,-175261.126629,-217715.071648,16,528,139.040000
16,18,-187102.076100,-217667.193495,37,529,193.111111
17,19,-184803.853689,-217067.375030,41,529,182.947368
19,21,-181172.092216,-216832.421450,14,529,165.523810
25,27,-170783.073271,-216633.826182,12,528,128.740741
20,22,-179174.544832,-216533.278011,22,528,158.000000
26,28,-167240.873909,-214790.030766,13,527,124.142857
13,15,-188952.289078,-214422.194634,49,529,231.733333
12,14,-190374.031683,-214145.533293,63,529,248.285714


In [ ]:
dist_renda = pd.read_sql_table("dist_renda_T", con=engine, schema=schema)

dist_renda = dist_renda.fillna(0)

# dist_renda = dist_renda.drop(columns=["Total"])

## dist_instrucao

In [87]:
dist_instucao_gen = pd.read_sql_table("dist_instrucao_genero", con=engine, schema=schema)

In [88]:
dist_instucao_gen

,id,"Brasil, Grande Região, Unidade da Federação e Município",populacao,Sem instrução e fundamental incompleto,Fundamental completo e médio incompleto,Médio completo e superior incompleto,Superior completo,Cod
0,499,Arame,total,572.912801,184.291899,181.570810,61.224490,2100956
1,14416,Araruama,mulheres,321.404482,152.308326,382.803717,143.464738,3300209
2,16674,Iporá,mulheres,343.000827,131.716726,303.182695,222.030862,5210208
3,5787,Abaetetuba,homens,442.114169,142.000756,350.213325,65.671749,1500107
4,11390,Abaetetuba,mulheres,347.044934,136.382197,396.612637,119.960232,1500107
...,...,...,...,...,...,...,...,...
5614,14521,Álvares Machado,mulheres,294.031356,174.291739,357.385165,174.291739,3501301
5615,3315,Álvares Machado,total,291.491597,182.056914,371.848739,154.602750,3501301
5616,5868,Óbidos,homens,523.944140,205.518177,239.667552,30.870131,1505106
5617,11471,Óbidos,mulheres,406.923540,180.012041,327.393137,85.731487,1505106


In [ ]:
dist_instucao_total = dist_instucao_gen[dist_instucao_gen["populacao"] == "total"].copy()
dist_instucao_restante = dist_instucao_gen[dist_instucao_gen["populacao"] != "total"].copy()

In [92]:
dist_instucao_gen = pivot_multiplas_colunas_populacao(
    df= dist_instucao_restante,
    categoria_col="populacao",
    populacao_cols=['Sem instrução e fundamental incompleto',
       'Fundamental completo e médio incompleto',
       'Médio completo e superior incompleto', 'Superior completo'],
    rename_categories={
        "homens": "homens",
        "mulheres": "mulheres",
    },
)

In [91]:
dist_instucao_total.columns

Index(['id', 'Brasil, Grande Região, Unidade da Federação e Município',
       'populacao', 'Sem instrução e fundamental incompleto',
       'Fundamental completo e médio incompleto',
       'Médio completo e superior incompleto', 'Superior completo', 'Cod'],
      dtype='str')

In [95]:
stats, custers = test_gmm(
   df = dist_instucao_total, 
   reference_column="Cod",
   coordinate_columns=['Sem instrução e fundamental incompleto',
       'Fundamental completo e médio incompleto',
       'Médio completo e superior incompleto', 'Superior completo'],
   k_min=2, k_max=max_clusters
)

best_bic_row = stats.loc[stats["bic"].idxmin()]
best_aic_row = stats.loc[stats["aic"].idxmin()]

print(f"menor BIC: {best_bic_row['bic']} | k = {best_bic_row['k']}")
print(f"menor AIC: {best_aic_row['aic']} | k = {best_aic_row['k']}")

stats.sort_values("aic")

testando para 2 clusters
testando para 3 clusters
testando para 4 clusters
testando para 5 clusters
testando para 6 clusters
testando para 7 clusters
testando para 8 clusters
testando para 9 clusters
testando para 10 clusters
testando para 11 clusters
testando para 12 clusters
testando para 13 clusters
testando para 14 clusters
testando para 15 clusters
testando para 16 clusters
testando para 17 clusters
testando para 18 clusters
testando para 19 clusters
testando para 20 clusters
testando para 21 clusters
testando para 22 clusters
testando para 23 clusters
testando para 24 clusters
testando para 25 clusters
testando para 26 clusters
testando para 27 clusters
testando para 28 clusters
testando para 29 clusters
testando para 30 clusters
menor BIC: 45407.31177094128 | k = 6.0
menor AIC: 44348.888524174996 | k = 23.0


,k,bic,aic,min_cluster_size,max_cluster_size,mean_cluster_size
21,23,46253.030590,44348.888524,10,549,81.434783
28,30,46861.111453,44375.763234,5,452,62.433333
25,27,46616.714722,44380.454854,4,564,69.370370
22,24,46380.850725,44393.679209,7,579,78.041667
27,29,46821.058105,44418.739336,4,466,64.586207
24,26,46572.741669,44419.511251,4,487,72.038462
26,28,46745.411974,44426.122656,6,426,66.892857
16,18,45924.205133,44435.210320,5,495,104.055556
18,20,46097.148545,44442.094831,9,481,93.650000
20,22,46267.173601,44446.060986,4,490,85.136364


In [94]:
dist_instucao_gen.columns

Index(['id', 'Brasil, Grande Região, Unidade da Federação e Município', 'Cod',
       'Fundamental completo e médio incompleto_homens',
       'Fundamental completo e médio incompleto_mulheres',
       'Médio completo e superior incompleto_homens',
       'Médio completo e superior incompleto_mulheres',
       'Sem instrução e fundamental incompleto_homens',
       'Sem instrução e fundamental incompleto_mulheres',
       'Superior completo_homens', 'Superior completo_mulheres'],
      dtype='str')

In [96]:
stats, custers = test_gmm(
   df = dist_instucao_gen, 
   reference_column="Cod",
   coordinate_columns=['Fundamental completo e médio incompleto_homens',
       'Fundamental completo e médio incompleto_mulheres',
       'Médio completo e superior incompleto_homens',
       'Médio completo e superior incompleto_mulheres',
       'Sem instrução e fundamental incompleto_homens',
       'Sem instrução e fundamental incompleto_mulheres',
       'Superior completo_homens', 'Superior completo_mulheres'],
   k_min=2, k_max=max_clusters
)

best_bic_row = stats.loc[stats["bic"].idxmin()]
best_aic_row = stats.loc[stats["aic"].idxmin()]

print(f"menor BIC: {best_bic_row['bic']} | k = {best_bic_row['k']}")
print(f"menor AIC: {best_aic_row['aic']} | k = {best_aic_row['k']}")

stats.sort_values("aic")

testando para 2 clusters
testando para 3 clusters
testando para 4 clusters
testando para 5 clusters
testando para 6 clusters
testando para 7 clusters
testando para 8 clusters
testando para 9 clusters
testando para 10 clusters
testando para 11 clusters
testando para 12 clusters
testando para 13 clusters
testando para 14 clusters
testando para 15 clusters
testando para 16 clusters
testando para 17 clusters
testando para 18 clusters
testando para 19 clusters
testando para 20 clusters
testando para 21 clusters
testando para 22 clusters
testando para 23 clusters
testando para 24 clusters
testando para 25 clusters
testando para 26 clusters
testando para 27 clusters
testando para 28 clusters
testando para 29 clusters
testando para 30 clusters
menor BIC: -77138.93729970285 | k = 7.0
menor AIC: -80054.38277507952 | k = 22.0


,k,bic,aic,min_cluster_size,max_cluster_size,mean_cluster_size
20,22,-73894.451775,-80054.382775,15,793,170.272727
25,27,-72412.645552,-79973.976426,10,584,138.740741
12,14,-76042.562483,-79960.253686,30,1140,267.571429
13,15,-75758.217058,-79956.188235,30,1141,249.733333
26,28,-72104.047729,-79945.658577,10,590,133.785714
14,16,-75453.022605,-79931.273757,29,1139,234.125000
22,24,-73192.779213,-79913.270162,14,592,156.083333
16,18,-74864.508408,-79903.319509,29,1139,208.111111
18,20,-74255.810744,-79855.181795,33,1139,187.300000
24,26,-72562.816014,-79843.866914,15,947,144.076923
